In [0]:
%sql
WITH metricas_generales AS (
SELECT
  COUNT(*) AS total_registros,
  COUNT(DISTINCT zona) AS propiedades_unicas,
  COUNT(DISTINCT tipo_de_operacion) AS tipo_de_operacion,
  ROUND(AVG(ambientes),2) AS ambiente_promedio
FROM bronze_EDA
),
problemas_calidad AS (
  SELECT
  ROUND(SUM(CASE WHEN precio IS NULL OR precio <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),2) AS pct_precio_invalido,

  ROUND(SUM(CASE WHEN metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),2) AS pct_m2_totales_invalido,

  ROUND(SUM(CASE WHEN antiguedad = '999' OR CAST(antiguedad AS float) = 999 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),2) AS pct_antiguedad_999
FROM bronze_EDA
)
SELECT
'RESUMEN DEL DATASET' AS seccion,
  A.total_registros,
  A.propiedades_unicas,
  A.tipo_de_operacion,
  A.ambiente_promedio,
  B.pct_precio_invalido || '%' AS precio_invalido,
  B.pct_m2_totales_invalido || '%' AS m2_totales_invalido,
  B.pct_antiguedad_999 || '%' AS antiguedad_999
FROM metricas_generales A
CROSS JOIN problemas_calidad B

## 📋 Hallazgos del EDA

### Problemas de calidad identificados:

1. **Valores placeholder:** El valor `999` en antigüedad representa datos faltantes
2. **Campos nulos:** Varios campos como `expensas`, `orientacion`, `cochera` tienen alto % de nulos
3. **Datos mixtos:** Precios en USD y ARS mezclados (requiere normalización)
4. **Posibles duplicados:** Propiedades repetidas con misma ubicación y precio
5. **Outliers:** Precios extremadamente altos o bajos que podrían ser errores
6. **Texto sucio:** La columna `original_text` tiene caracteres mal codificados

### Transformaciones necesarias para capa Silver:

- [ ] Convertir NULL en antigüedad a `999`
- [ ] Normalizar precios a una sola moneda (o crear columnas separadas)
- [ ] Eliminar duplicados
- [ ] Filtrar outliers extremos
- [ ] Limpiar caracteres especiales en texto
- [ ] Parsear el campo ubicación para extraer barrio/ciudad
- [ ] Calcular métricas derivadas (precio por m2)
